In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Estimator로 노이즈 관리 구성

<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
노이즈를 관리하는 방법에는 여러 가지가 있으며, 일반적으로 오류가 발생하기 전에 방지하기 위해 다양한 오류 완화 및 오류 억제 기법을 사용합니다. 이러한 기법은 보통 전처리 오버헤드를 유발합니다. 따라서 결과를 완벽히 하는 것과 Job이 적절한 시간 내에 완료되도록 하는 것 사이의 균형을 달성하는 것이 중요합니다.

Estimator는 다음과 같은 노이즈 관리 기법을 지원합니다. 각 기법에 대한 설명은 [오류 완화 및 억제 기법](error-mitigation-and-suppression-techniques)을 참조하세요. 이러한 기법을 활성화하는 방법은 [사용자 정의 오류 설정](#advanced-error) 섹션을 참조하세요.

- [동적 분리](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli 트월링](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## 복원력 수준
`resilience_level`은 오류에 대한 복원력을 얼마나 구축할지를 지정합니다. 높은 수준은 더 긴 처리 시간을 대가로 더 정확한 결과를 생성합니다. 복원력 수준은 Primitive 쿼리에 노이즈 관리를 적용할 때 비용/정확도 트레이드오프를 구성하는 데 사용할 수 있습니다. 노이즈 관리는 관련 Circuit 모음 또는 앙상블의 출력을 처리하여 결과의 오류(편향)를 줄입니다. 오류 감소 정도는 적용된 방법에 따라 다릅니다. 복원력 수준은 사용자가 자신의 애플리케이션에 적합한 비용/정확도 트레이드를 추론할 수 있도록 노이즈 관리 방법의 세부적인 선택을 추상화합니다.

이를 감안하면, 각 수준은 다양한 시간-정확도 트레이드오프를 실험할 수 있도록 양자 샘플링 오버헤드 수준이 증가하는 방법 또는 방법들에 해당합니다. 다음 표는 각 Primitive에 대해 어떤 수준과 해당 방법이 사용 가능한지 보여줍니다.

<span id="resilience-table"></span>

| 복원력 수준 | 설명                                                                                            | 기법                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | 완화 없음                                                                                         | 없음                                                                      |
| 1 [기본값]      | 최소 완화 비용: 읽기 오류와 관련된 오류 완화               | [트월된 읽기 오류 소거 (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) 측정 트월링             |
| 2                | 중간 완화 비용. 일반적으로 Estimator의 편향을 줄이지만 제로 편향이 보장되지는 않습니다. | 수준 1 + [제로 노이즈 외삽 (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) 및 게이트 트월링

> **Info:** 복원력 수준은 현재 베타 상태이므로 샘플링 오버헤드와 솔루션 품질은 Circuit마다 다를 수 있습니다. 새로운 기능, 고급 옵션 및 관리 도구는 순차적으로 출시될 예정입니다. 특정 노이즈 관리 방법이 각 복원력 수준에서 적용된다는 보장이 없습니다.

### 복원력 수준으로 Estimator 구성
복원력 수준을 사용하여 노이즈 관리 기법을 지정하거나, [사용자 정의 오류 설정](#advanced-error)에 설명된 것처럼 개별적으로 사용자 정의 기법을 설정할 수 있습니다.

> **Note:** 복원력 수준에 추가하여 수동으로 지정하는 옵션은 복원력 수준에 의해 정의된 기본 옵션 세트에 추가로 적용됩니다. 따라서 원칙적으로 복원력 수준을 1로 설정하고 측정 완화를 끌 수 있지만, 이는 권장하지 않습니다.
> 
> 예를 들어, 복원력 수준을 0으로 설정하면 `zne_mitigation`이 꺼지지만, `estimator.options.resilience.zne_mitigation = True`는 해당 값을 재정의합니다.

### 예제
다음 코드는 `resilience_level 2`를 설정하여 ZNE, TREX 및 게이트 트월링을 활성화합니다.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## 사용자 정의 노이즈 관리 설정
[Estimator 옵션](/guides/estimator-options)을 사용하여 개별 노이즈 관리 방법을 켜고 끌 수 있습니다.

> **Note:** 모든 옵션이 모든 유형의 Circuit에서 함께 작동하지는 않습니다. 자세한 내용은 [기능 호환성 표](estimator-options#options-compatibility-table)를 참조하세요.

### 예제

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>